# SAR-to-EO Image Translation — Kaggle Training Notebook

**GalaxEye Technical Assignment.** This notebook runs the whole pipeline end-to-end on Kaggle's free **T4 GPU**:
data prep → train the full Pix2Pix model → train the L1-only ablation → evaluate both on the val & test splits → save loss curves and qualitative triplets.

### Before you run
1. Top-right **⋮ → Accelerator → GPU T4 x1**.
2. **+ Add Input** → search `sentinel12-image-pairs-segregated-by-terrain` (by *requiemonk*) → Add.
3. Set `REPO_URL` below to your **public GitHub repo** (the one you'll submit). The notebook clones it.
4. **Run All**. First a quick *smoke test* (few epochs, subset) verifies everything, then you scale up.


## 1. Settings

In [1]:
# ---- EDIT THESE ----
REPO_URL = 'https://github.com/codehunter17/sar2eo.git'  # your public repo
VAL_TERRAIN  = 'grassland'   # terrain held out for validation
TEST_TERRAIN = 'urban'       # terrain held out for test

# Smoke test first (fast, proves the pipeline). Set SMOKE=False for the real run.
SMOKE = False
SMOKE_EPOCHS = 60
SMOKE_SUBSET = 2000     # pairs per terrain during smoke test
FULL_EPOCHS  = 40     # real-run epochs (matches config.yaml)
# --------------------
import os, glob, sys, subprocess

# Auto-locate the attached Kaggle dataset (the folder that holds the terrain dirs)
cands = glob.glob('/kaggle/input/*sentinel*') + glob.glob('/kaggle/input/*')
DATA_SRC = next((c for c in cands if os.path.isdir(c)), None)
print('Dataset input dir :', DATA_SRC)
assert DATA_SRC, 'No /kaggle/input dataset found — did you Add the dataset as Input?'


Dataset input dir : /kaggle/input/datasets


## 2. Get the code

In [2]:
WORK = '/kaggle/working/sar2eo'
if REPO_URL and '<your-username>' not in REPO_URL:
    if not os.path.exists(WORK):
        subprocess.run(['git','clone','--depth','1',REPO_URL,WORK], check=True)
else:
    # Fallback: code uploaded as a Kaggle dataset/input — copy it in.
    src_code = next((c for c in glob.glob('/kaggle/input/*') if os.path.exists(os.path.join(c,'train.py'))), None)
    assert src_code, 'Set REPO_URL, or attach your code as an Input dataset containing train.py.'
    subprocess.run(['cp','-r',src_code,WORK], check=True)
os.chdir(WORK)
print('Working dir:', os.getcwd())
print(os.listdir('.'))


Cloning into '/kaggle/working/sar2eo'...


Working dir: /kaggle/working/sar2eo
['KAGGLE_GUIDE.md', 'config.yaml', 'infer.py', 'train.py', 'data', 'Technical_Report.pdf', 'prepare_data.py', 'utils', 'README.md', 'REPORT.md', 'eval.py', 'models', '.gitignore', 'sar2eo_kaggle.ipynb', 'requirements.txt', '.git']


## 3. Dependencies
Kaggle already has PyTorch + CUDA. We only add the metric/util packages.

In [3]:
!pip -q install lpips==0.1.4 pytorch-fid==0.3.0 torchmetrics==1.3.2 pyyaml==6.0.1 tqdm==4.66.2
import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 841.5/841.5 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 725.0/725.0 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 104.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 4.8.5 requires tqdm>=4.66.3, but you have tqdm 4.66.2 which is incompatible.
featuretools 1.31.0 requires tqdm>=4.66.3, but you have tqdm 4.66.2 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 whic

## 4. Prepare data (terrain → train/val/test)

In [4]:
subset = ['--max_per_terrain', str(SMOKE_SUBSET)] if SMOKE else []
cmd = ['python','prepare_data.py','--src',DATA_SRC,'--dst','dataset',
       '--val_terrain',VAL_TERRAIN,'--test_terrain',TEST_TERRAIN] + subset
print(' '.join(cmd))
subprocess.run(cmd, check=True)


python prepare_data.py --src /kaggle/input/datasets --dst dataset --val_terrain grassland --test_terrain urban
[prepare_data] Found 4 terrain folder(s): ['agri', 'barrenland', 'grassland', 'urban']
[prepare_data] Held-out validation terrain: 'grassland'
[prepare_data] Held-out test terrain:       'urban'
[prepare_data] Training terrains: ['agri', 'barrenland']
  - agri           -> train : 4000 pairs
  - barrenland     -> train : 4000 pairs
  - grassland      -> val   : 4000 pairs
  - urban          -> test  : 4000 pairs
--------------------------------------------------------
[prepare_data] DONE.  Train: 8000 | Val: 4000 | Test: 4000 pairs
[prepare_data] Layout written under: /kaggle/working/sar2eo/dataset
[prepare_data] Set  data.data_root: "dataset"  in config.yaml


CompletedProcess(args=['python', 'prepare_data.py', '--src', '/kaggle/input/datasets', '--dst', 'dataset', '--val_terrain', 'grassland', '--test_terrain', 'urban'], returncode=0)

## 5. Build the two config variants (ablation)
Identical settings except the **loss** and the output folder.

In [5]:
import yaml, copy
base = yaml.safe_load(open('config.yaml'))
epochs = SMOKE_EPOCHS if SMOKE else FULL_EPOCHS

def write_cfg(path, mode, out_dir):
    c = copy.deepcopy(base)
    c['loss']['mode'] = mode
    c['experiment']['output_dir'] = out_dir
    c['training']['epochs'] = epochs
    if not SMOKE:
        c['training']['lr_decay_start_epoch'] = epochs // 2
    c['data']['data_root'] = 'dataset'
    c['training']['num_workers'] = 2
    yaml.safe_dump(c, open(path,'w'), sort_keys=False)
    print('wrote', path, '| mode=', mode, '| epochs=', epochs, '| out=', out_dir)

write_cfg('config_full.yaml', 'full_gan', 'outputs_full')
write_cfg('config_l1.yaml',   'l1_only',  'outputs_l1')


wrote config_full.yaml | mode= full_gan | epochs= 40 | out= outputs_full
wrote config_l1.yaml | mode= l1_only | epochs= 40 | out= outputs_l1


## 6. Train — Model B (full Pix2Pix: L1 + adversarial)

In [6]:
!python train.py --config config_full.yaml


[Train] Using device: cuda
[Dataset] Train: 8000 pairs | Val: 4000 pairs
[Train] Loss mode: full_gan
  Epoch   1 | Step   50 | G: 43.2344 | D: 0.3418
  Epoch   1 | Step  100 | G: 40.1949 | D: 0.2551
  Epoch   1 | Step  150 | G: 38.7008 | D: 0.2096
  Epoch   1 | Step  200 | G: 37.9868 | D: 0.1664
  Epoch   1 | Step  250 | G: 37.4528 | D: 0.1457
  Epoch   1 | Step  300 | G: 36.9654 | D: 0.1385
  Epoch   1 | Step  350 | G: 36.7690 | D: 0.1258
  Epoch   1 | Step  400 | G: 36.3410 | D: 0.1180
  Epoch   1 | Step  450 | G: 36.2147 | D: 0.1102
  Epoch   1 | Step  500 | G: 35.9640 | D: 0.1024
  Epoch   1 | Step  550 | G: 35.9064 | D: 0.0968
  Epoch   1 | Step  600 | G: 35.7364 | D: 0.0951
  Epoch   1 | Step  650 | G: 35.5310 | D: 0.0895
  Epoch   1 | Step  700 | G: 35.5431 | D: 0.0844
  Epoch   1 | Step  750 | G: 35.4016 | D: 0.0800
  Epoch   1 | Step  800 | G: 35.2484 | D: 0.0765
  Epoch   1 | Step  850 | G: 35.1779 | D: 0.0730
  Epoch   1 | Step  900 | G: 35.1182 | D: 0.0696
  Epoch   1 | Ste

## 7. Train — Model A (L1-only ablation)

In [7]:
!python train.py --config config_l1.yaml


[Train] Using device: cuda
[Dataset] Train: 8000 pairs | Val: 4000 pairs
[Train] Loss mode: l1_only
  Epoch   1 | Step   50 | G: 0.4269 | D: 0.0000
  Epoch   1 | Step  100 | G: 0.3952 | D: 0.0000
  Epoch   1 | Step  150 | G: 0.3789 | D: 0.0000
  Epoch   1 | Step  200 | G: 0.3706 | D: 0.0000
  Epoch   1 | Step  250 | G: 0.3640 | D: 0.0000
  Epoch   1 | Step  300 | G: 0.3585 | D: 0.0000
  Epoch   1 | Step  350 | G: 0.3559 | D: 0.0000
  Epoch   1 | Step  400 | G: 0.3514 | D: 0.0000
  Epoch   1 | Step  450 | G: 0.3500 | D: 0.0000
  Epoch   1 | Step  500 | G: 0.3474 | D: 0.0000
  Epoch   1 | Step  550 | G: 0.3467 | D: 0.0000
  Epoch   1 | Step  600 | G: 0.3451 | D: 0.0000
  Epoch   1 | Step  650 | G: 0.3431 | D: 0.0000
  Epoch   1 | Step  700 | G: 0.3431 | D: 0.0000
  Epoch   1 | Step  750 | G: 0.3415 | D: 0.0000
  Epoch   1 | Step  800 | G: 0.3399 | D: 0.0000
  Epoch   1 | Step  850 | G: 0.3391 | D: 0.0000
  Epoch   1 | Step  900 | G: 0.3384 | D: 0.0000
  Epoch   1 | Step  950 | G: 0.3375 

## 8. Evaluate both models on val and test splits
Runs inference then computes LPIPS / FID / SSIM / PSNR. Results saved as JSON.

In [8]:
import json
runs = {'full':'outputs_full', 'l1':'outputs_l1'}
results = {}
for tag, out in runs.items():
    w = f'{out}/final_weights.pth'
    for split in ['val','test']:
        pred = f'{out}/{split}_preds'
        subprocess.run(['python','infer.py','--input_dir',f'dataset/{split}/s1',
                        '--output_dir',pred,'--weights',w], check=True)
        mjson = f'{out}/metrics_{split}.json'
        subprocess.run(['python','eval.py','--pred_dir',pred,
                        '--gt_dir',f'dataset/{split}/s2','--save_json',mjson], check=True)
        results[(tag,split)] = json.load(open(mjson))

# Pretty summary table
print('\n=== RESULTS (paste into README + report) ===')
hdr = f"{'Model':<22}{'Split':<6}{'LPIPS':>9}{'FID':>9}{'SSIM':>9}{'PSNR':>9}"
print(hdr); print('-'*len(hdr))
names = {'full':'Pix2Pix (L1+GAN)', 'l1':'L1 only (ablation)'}
for tag in ['l1','full']:
    for split in ['val','test']:
        m = results[(tag,split)]
        print(f"{names[tag]:<22}{split:<6}{m['LPIPS']:>9.4f}{m['FID']:>9.2f}{m['SSIM']:>9.4f}{m['PSNR']:>9.2f}")


[infer] Device: cuda
[infer] Loading weights from: outputs_full/final_weights.pth
[infer] Generator loaded successfully.
[infer] Processing 4000 patches...
  [100/4000] grassland_00099.png → grassland_00099.png
  [200/4000] grassland_00199.png → grassland_00199.png
  [300/4000] grassland_00299.png → grassland_00299.png
  [400/4000] grassland_00399.png → grassland_00399.png
  [500/4000] grassland_00499.png → grassland_00499.png
  [600/4000] grassland_00599.png → grassland_00599.png
  [700/4000] grassland_00699.png → grassland_00699.png
  [800/4000] grassland_00799.png → grassland_00799.png
  [900/4000] grassland_00899.png → grassland_00899.png
  [1000/4000] grassland_00999.png → grassland_00999.png
  [1100/4000] grassland_01099.png → grassland_01099.png
  [1200/4000] grassland_01199.png → grassland_01199.png
  [1300/4000] grassland_01299.png → grassland_01299.png
  [1400/4000] grassland_01399.png → grassland_01399.png
  [1500/4000] grassland_01499.png → grassland_01499.png
  [1600/4000]

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


SAR-to-EO Evaluation
  Predictions : outputs_full/val_preds
  Ground truth: dataset/val/s2

Computing PSNR and SSIM...
Computing LPIPS...
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 165MB/s]


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
Computing FID...
Downloading: "https://github.com/mseitzer/pytorch-fid/releases/download/fid_weights/pt_inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/pt_inception-2015-12-05-6726825d.pth


...


Results
  LPIPS (↓ better): 0.7896
  FID   (↓ better): 285.22
  SSIM  (↑ better): 0.2531
  PSNR  (↑ better): 14.1 dB
  Results saved → outputs_l1/metrics_val.json
[infer] Device: cuda
[infer] Loading weights from: outputs_l1/final_weights.pth
[infer] Generator loaded successfully.
[infer] Processing 4000 patches...
  [100/4000] urban_00099.png → urban_00099.png
  [200/4000] urban_00199.png → urban_00199.png
  [300/4000] urban_00299.png → urban_00299.png
  [400/4000] urban_00399.png → urban_00399.png
  [500/4000] urban_00499.png → urban_00499.png
  [600/4000] urban_00599.png → urban_00599.png
  [700/4000] urban_00699.png → urban_00699.png
  [800/4000] urban_00799.png → urban_00799.png
  [900/4000] urban_00899.png → urban_00899.png
  [1000/4000] urban_00999.png → urban_00999.png
  [1100/4000] urban_01099.png → urban_01099.png
  [1200/4000] urban_01199.png → urban_01199.png
  [1300/4000] urban_01299.png → urban_01299.png
  [1400/4000] urban_01399.png → urban_01399.png
  [1500/4000] urban

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


SAR-to-EO Evaluation
  Predictions : outputs_l1/test_preds
  Ground truth: dataset/test/s2

Computing PSNR and SSIM...
Computing LPIPS...
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
Computing FID...


100%|██████████| 125/125 [00:23<00:00,  5.42it/s]



Results
  LPIPS (↓ better): 0.739
  FID   (↓ better): 386.3
  SSIM  (↑ better): 0.1402
  PSNR  (↑ better): 12.66 dB
  Results saved → outputs_l1/metrics_test.json

=== RESULTS (paste into README + report) ===
Model                 Split     LPIPS      FID     SSIM     PSNR
----------------------------------------------------------------
L1 only (ablation)    val      0.7896   285.22   0.2531    14.10
L1 only (ablation)    test     0.7390   386.30   0.1402    12.66
Pix2Pix (L1+GAN)      val      0.8006   280.63   0.2483    13.86
Pix2Pix (L1+GAN)      test     0.7327   384.55   0.1393    12.71


## 9. Collect deliverables for the ZIP
Gathers loss curves + qualitative triplets into one folder you can download.

In [9]:
import shutil, os
os.makedirs('submission_assets', exist_ok=True)
for out in ['outputs_full','outputs_l1']:
    if os.path.exists(f'{out}/loss_curve.png'):
        shutil.copy(f'{out}/loss_curve.png', f'submission_assets/loss_curve_{out}.png')
    if os.path.exists(f'{out}/loss_log.csv'):
        shutil.copy(f'{out}/loss_log.csv', f'submission_assets/loss_log_{out}.csv')
    sdir = f'{out}/samples'
    if os.path.isdir(sdir):
        dst = f'submission_assets/samples_{out}'; os.makedirs(dst, exist_ok=True)
        for f in sorted(os.listdir(sdir))[-8:]:
            shutil.copy(os.path.join(sdir,f), os.path.join(dst,f))
print('Assets ready in submission_assets/:')
for f in sorted(os.listdir('submission_assets')): print(' -', f)
print('\nWeights to upload publicly:')
print(' - outputs_full/final_weights.pth  (this is the model you submit)')


Assets ready in submission_assets/:
 - loss_curve_outputs_full.png
 - loss_curve_outputs_l1.png
 - loss_log_outputs_full.csv
 - loss_log_outputs_l1.csv
 - samples_outputs_full
 - samples_outputs_l1

Weights to upload publicly:
 - outputs_full/final_weights.pth  (this is the model you submit)


## 10. Next steps after this finishes
1. Check the smoke-test ran clean, then set `SMOKE = False` (cell 1) and **Run All** again for the real run.
2. **Download** `outputs_full/final_weights.pth` and `submission_assets/` (Output tab → download).
3. **Upload weights** to Hugging Face Hub (public) → put the link in README + the submission form.
4. **Fill in** the `[fill after training]` cells in `Technical_Report.pdf`/`REPORT.md` and the README results table using the printed metrics, loss curves, and 5 triplets.
5. Commit everything to your public GitHub repo and build `Krishna_Kant_GalaxEye.zip` (report PDF + time log + loss curves + triplet images; **not** the weights).
